# back-fn-call-with-recipe-args — ex2: diagnose back_fn call-site bugs via a recording wrapper

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `back-fn-call-with-recipe-args`. Running the final beacon cell reports progress against the `Backprop: back fn call with recipe args` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
# === manual autograd primitives — shared across all drills in this folder ===
from dataclasses import dataclass, field
from typing import Any, Callable, Optional

grad_tracking_enabled = True

@dataclass
class Recipe:
    func: Optional[Callable] = None
    args: tuple = ()
    kwargs: dict = field(default_factory=dict)
    parents: dict = field(default_factory=dict)

class MiniTensor:
    """A minimal Tensor wrapper for the ARENA-style manual-autograd drills.
    Wraps a raw `torch.Tensor` in `.array`. Carries an optional `.recipe`
    populated by wrap_forward_fn. `requires_grad` is set by the wrapper.
    `.grad` accumulates the leaf gradient at the end of the reverse pass."""
    def __init__(self, array, requires_grad: bool = False, recipe=None):
        self.array = array
        self.requires_grad = requires_grad
        self.recipe = recipe
        self.grad = None
    def __repr__(self):
        return f'MiniTensor({self.array!r}, requires_grad={self.requires_grad})'

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Backprop: back fn call with recipe args` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`back-fn-call-with-recipe-args`** (exercise 2). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "back-fn-call-with-recipe-args"
DD_SUBTOPIC = "Backprop: back fn call with recipe args"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Back_fn call channels — splat-bug spy — quick refresher

The canonical call:
```
back_fn(grad_out, node.array, *node.recipe.args, **node.recipe.kwargs)
```
has FOUR channels, each of which can be silently miswired:

1. `grad_out` — dL/d(out); missing it = no chain-rule multiplier.
2. `node.array` — cached forward `out`; passing `node` (the wrapper) instead breaks back_fns that do tensor math on `out`.
3. `*recipe.args` — forgetting the `*` passes the whole tuple as a single argument; back_fns crash on signature mismatch.
4. `**recipe.kwargs` — forgetting the `**` drops `dim`/`keepdim`/etc.; reductions silently use the wrong axis.

A **spy back_fn** that records every received argument lets you diagnose which channel went wrong — recording (grad_out, out, args, kwargs) and comparing against expected, an authoring helper can report 'kwargs={} but recipe.kwargs={'dim': 1}' — pinpointing the missing `**` splat.

### Exercise 2 — diagnose back_fn call-site bugs via a recording wrapper

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Bloom level: Analyze
> LO: Analyze a back_fn invocation by writing a `call_back_fn_recording` wrapper that returns the back_fn result PLUS a dict capturing exactly what reached each of the four channels (grad_out, out, args, kwargs) — enabling diagnosis of missing *, **, or wrong `out` passes.
> Keywords: debug, splat-bug, back-fn-call, spy, recording
> ```

**KCs targeted:** `back-fn-call-with-recipe-args`, `kwargs-pass-through-recipe`

Implement `call_back_fn_recording(back_fn, grad_out, node)` — the canonical invocation from ex1 PLUS an audit trail. Returns `(result, record)` where:

- `result` is whatever the back_fn returned (a `torch.Tensor`).
- `record` is a dict with FOUR keys:
  * `'grad_out'` — the `grad_out` you forwarded.
  * `'out'` — the second positional you forwarded (should be     `node.array`, NOT `node`).
  * `'args'` — the tuple of positional args after `out` (should     be `node.recipe.args`).
  * `'kwargs'` — the kwargs dict (should be `node.recipe.kwargs`).

**How to record.** Wrap the back_fn in a one-shot closure that captures every arg it receives BEFORE forwarding to the real back_fn. The simplest way:

```python
record = {}
def _spy(g_out, out_, *args, **kwargs):
    record['grad_out'] = g_out
    record['out'] = out_
    record['args'] = args
    record['kwargs'] = kwargs
    return back_fn(g_out, out_, *args, **kwargs)
result = _spy(grad_out, node.array, *node.recipe.args, **node.recipe.kwargs)
```

The recorded dict is the diagnostic surface — a caller can diff `record['kwargs']` against `node.recipe.kwargs` to see if any kwarg got dropped (missing `**` splat), check `record['out'] is node.array` to confirm the `out` channel is the raw tensor, etc.

**Why a wrapper instead of just calling and returning result.** ex1 returned only the result. ex2 elevates the call site into an analysis tool — the same invocation, but every channel is now externally inspectable. Useful when porting a back_fn from one autograd to another, or when a backward pass produces unexpectedly-shaped grads.

In [ ]:
def call_back_fn_recording(back_fn, grad_out, node) -> tuple:
    record = {}

    def _spy(g_out, out_, *args, **kwargs):
        # Record EVERY channel before forwarding to the real back_fn.
        record['grad_out'] = g_out
        record['out'] = out_
        record['args'] = args
        record['kwargs'] = kwargs
        return back_fn(g_out, out_, *args, **kwargs)

    # Canonical invocation — SAME shape as ex1, just through the spy.
    result = _spy(
        grad_out,
        node.array,                # raw torch.Tensor, NOT the MiniTensor
        *node.recipe.args,         # forward positional args (unboxed)
        **node.recipe.kwargs,      # forward kwargs (dim, keepdim, ...)
    )
    return result, record


<details><summary>Solution</summary>

```python
def call_back_fn_recording(back_fn, grad_out, node) -> tuple:
    record = {}

    def _spy(g_out, out_, *args, **kwargs):
        # Record EVERY channel before forwarding to the real back_fn.
        record['grad_out'] = g_out
        record['out'] = out_
        record['args'] = args
        record['kwargs'] = kwargs
        return back_fn(g_out, out_, *args, **kwargs)

    # Canonical invocation — SAME shape as ex1, just through the spy.
    result = _spy(
        grad_out,
        node.array,                # raw torch.Tensor, NOT the MiniTensor
        *node.recipe.args,         # forward positional args (unboxed)
        **node.recipe.kwargs,      # forward kwargs (dim, keepdim, ...)
    )
    return result, record
```

**The spy is a one-line lens.** It doesn't change behavior — the back_fn still computes the same gradient — but every channel becomes externally inspectable. A test that compares `record['kwargs']` against `node.recipe.kwargs` will catch a missing `**` splat immediately; otherwise the bug only manifests when the back_fn shape-mismatches downstream.

**Why a closure, not a wrapper class.** A `class CallRecorder` would work but adds ceremony. The closure captures `record` by reference, mutates it on every call (here, just one), and is gone the moment the function returns — exactly the lifetime we want.

**Composability with the ex1 call_back_fn.** A tidy refactoring would have `call_back_fn` (no record) and `call_back_fn_recording` (with record) share an internal `_invoke(back_fn, grad_out, node, hook=None)` helper. For the drill, the duplication is small enough that we keep them side-by-side.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex2'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex2',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()